# 04 · PD model: champion vs challenger
**Champion:** logistic regression on WoE inputs (core features, **no LC grade/int_rate**).
**Challenger:** LightGBM with monotone constraints (FICO down, DTI/revol_util/delinquencies up, etc.).
**Benchmarks:** LC grade only; core + LC (`with_lc`) for incremental lift.

### Justifications
- **Logit-on-WoE as champion:** interpretable, monotone by construction, standard in credit, easy to validate. GBM is a *challenger*; it replaces the champion only if it earns it (below).
- **Excluding `int_rate`/`grade`/`sub_grade` from the champion:** known at origination (so not temporal leakage) but they encode **LC's own PD model** and pricing; a model that leans on them mostly re-learns LC's score. The `with_lc` row quantifies how much lift they add.
- **Class weights over SMOTE:** class weights rebalance the loss without inventing observations. Either way predicted PD levels are inflated, so we **recalibrate on train out-of-fold scores only** (notebook 05).
- **Monotone constraints in LightGBM:** keeps the challenger economically sensible and comparable to the scorecard; regularised (`min_child_samples=200`, `lambda=5`) to limit overfit to the 2007-14 regime.
- **Banned features** (`funded_amnt`, `out_prncp`, `recoveries`, ...) are blocked by `src/features/guards.py` and unit tests.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.pipeline import prepare, get_models, evaluate_models
data = prepare(cfg)
models = get_models(data, cfg)       # cached on a config hash; pass refit=True to retrain
champ = models["champion_logit"]
print("champion features:", champ.features)

**Champion coefficients** (expect all positive under the `ln(bad/good)` WoE convention) and **VIF**.

In [ ]:
from src.models.pd_models import logit_coef_table, vif_table
X = champ._X(data.train)
logit_coef_table(champ.model, X.columns).merge(vif_table(X), on="feature")

**Discrimination across models** (calibrated PD; AUC/Gini/KS are rank-based so identical to raw).

In [ ]:
perf = evaluate_models(models, data)
p = perf[perf.pd_type=="calibrated"].pivot_table(index="model", columns="sample", values=["auc","gini","ks"])
p

In [ ]:
from sklearn.metrics import roc_curve
yv = data.valid.target.astype(int)
fig, ax = plt.subplots(figsize=(5.5,5))
for n in ["grade_only_logit","champion_logit","with_lc_logit","challenger_lgbm"]:
    f,t,_ = roc_curve(yv, models[n].predict(data.valid)); ax.plot(f,t,label=n)
ax.plot([0,1],[0,1],"k:"); ax.legend(); ax.set(title="ROC, 2015 clean OOT validation", xlabel="FPR", ylabel="TPR"); plt.show()

**Decision rule (stated before looking at results):** the challenger replaces the champion only if validation Gini improves by **>= 0.02** *and* it is not materially worse on calibration. Otherwise the scorecard stays: equal power, more transparency.

In [ ]:
v = perf[(perf.pd_type=="calibrated") & (perf["sample"]=="valid_clean")].set_index("model")
uplift = v.loc["challenger_lgbm","gini"] - v.loc["champion_logit","gini"]
lc_lift = v.loc["with_lc_logit","gini"] - v.loc["champion_logit","gini"]
print(f"Challenger Gini uplift vs champion: {uplift:+.4f}  ->  {'CHALLENGER earns promotion' if uplift>=0.02 else 'KEEP champion'}")
print(f"Incremental Gini from adding LC int_rate/grade: {lc_lift:+.4f}")
tr_gini = perf[(perf.pd_type=="calibrated") & (perf["sample"]=="train (in-sample)")].set_index("model").gini
print("Train->valid Gini decay (larger = more overfit):")
for m_ in ["champion_logit", "challenger_lgbm"]:
    print(f"  {m_}: {tr_gini[m_] - v.loc[m_, 'gini']:.4f}")

**Challenger feature importance (gain)**.

In [ ]:
g = models["challenger_lgbm"]
imp = pd.Series(g.model.booster_.feature_importance("gain"), index=g.prep.numeric+g.prep.categorical).sort_values(ascending=False)
(imp/imp.sum()).to_frame("gain_share")

### SMOTE vs class weights (tested once, as a documented negative result)
SMOTE interpolates synthetic defaults in WoE space until defaults are ~50% of the training set. The model then believes defaults are common, so **predicted PDs are inflated relative to the true base rate**. Provision = PD x LGD x EAD is linear in PD, so the bias flows straight into the provision unless corrected. Class weights have the same *level* problem but it is a pure intercept shift that train-only recalibration removes; SMOTE additionally fabricates points, which can distort the slope as well as the level. Compare `over_prediction_ratio` below.

In [ ]:
from src.models.pd_models import smote_experiment
smote_experiment(champ._X(data.train), data.train.target, champ._X(data.valid), data.valid.target, cfg)

### Sensitivity: does dropping 2007-2011 (small volumes, different underwriting) change the story?

In [ ]:
from src.pipeline import PDModel
from src.evaluation.metrics import performance_row
sub = data.train[data.train.vintage_year.between(*cfg["split"]["sensitivity_train_years"])]
s_m = PDModel(cfg, champ.numeric, champ.categorical, "logit").fit(sub)
pd.DataFrame([performance_row(yv, champ.predict(data.valid), "champion | train 2007-14"),
              performance_row(yv, s_m.predict(data.valid), f"champion | train {cfg['split']['sensitivity_train_years']}")])

**Bootstrap CI on the champion-vs-challenger Gini gap.**

In [ ]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(cfg['seed'])
yv_arr = np.asarray(yv)
p_ch = models['champion_logit'].predict(data.valid)
p_cl = models['challenger_lgbm'].predict(data.valid)
B, n = 1000, len(yv_arr); deltas = np.empty(B)
for b in range(B):
    idx = rng.integers(0, n, n)
    deltas[b] = (2*roc_auc_score(yv_arr[idx], p_cl[idx]) - 1) \
              - (2*roc_auc_score(yv_arr[idx], p_ch[idx]) - 1)
lo, hi = np.percentile(deltas, [2.5, 97.5])
print(f'Gini gap = {deltas.mean():+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}]')